In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
import warnings

warnings.filterwarnings("ignore")

load_dotenv()

ROOT_DIR = os.getenv("DATA_DIR")

In [2]:
def create_train_test_split(input_path, output_dir, test_size=0.2):
    print("[INFO] Computing train test split...")
    df = pd.read_csv(input_path)

    # Computing unique global id for each student
    df['global_id'] = df['course_id'].astype(str) + "_" + df['student_id'].astype(str)

    # Binarize ground truth
    df['dropout'] = (df['dropout'] >= 0.5).astype(int)

    df_students = df.groupby('global_id').agg({
        'dropout': 'first',
        'course_id': 'first'
    }).reset_index()

    df_students['stratify_key'] = df_students['course_id'].astype(str) + "_" + df_students['dropout'].astype(str)

    # Handling difficoult courses
    key_counts = df_students['stratify_key'].value_counts()
    rare_keys = key_counts[key_counts < 2].index
    df_rare = df_students[df_students['stratify_key'].isin(rare_keys)]
    df_plentiful = df_students[~df_students['stratify_key'].isin(rare_keys)]

    print(f"[INFO] Found {len(df_rare)} students in small classes (forced into train)")

    # SSS only with min 2 samples
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=42)

    for train_idx, test_idx in sss.split(df_plentiful['global_id'], df_plentiful['stratify_key']):
        train_ids_plentiful = df_plentiful.iloc[train_idx]['global_id'].values
        test_ids = df_plentiful.iloc[test_idx]['global_id'].values

    train_ids = list(train_ids_plentiful) + list(df_rare['global_id'].values)

    df_train = df[df['global_id'].isin(train_ids)].drop(columns=['global_id', 'stratify_key'], errors='ignore')
    df_test = df[df['global_id'].isin(test_ids)].drop(columns=['global_id', 'stratify_key'], errors='ignore')

    os.makedirs(output_dir, exist_ok=True)
    df_train.to_csv(os.path.join(output_dir, "train_timeseries.csv"), index=False)
    df_test.to_csv(os.path.join(output_dir, "test_timeseries.csv"), index=False)

    print(f"[INFO] Saved train ({df_train.shape[0]} rows) | test ({df_test.shape[0]} rows)")

In [3]:
INPUT_FILE = os.path.join(ROOT_DIR, "processed/processed_all_timeseries.csv")
OUTPUT_DIR = os.path.join(ROOT_DIR, "train_test")

create_train_test_split(INPUT_FILE, OUTPUT_DIR)

[INFO] Computing train test split...
[INFO] Found 1 students in small classes (forced into train)
[INFO] Saved train (1485000 rows) | test (371160 rows)


In [4]:
# Head of outputs
df_train = pd.read_csv(os.path.join(OUTPUT_DIR, "train_timeseries.csv"))
df_test = pd.read_csv(os.path.join(OUTPUT_DIR, "test_timeseries.csv"))

df_train.head()

,student_id,course_id,day,dropout,view,write,user report,update,view forum,subscribe,...,blocked,unblocked,disabled,removed,accepted,assigned,restored,unassigned,abandoned,recent
0,0,1,1,1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,1,2,1,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,1,3,1,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0,1,4,1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,1,5,1,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
